In [9]:
import os
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
from src import config
#from src import bm25
from src import hyde
#from src import dpr
from src import tf_idf

import gc
import torch

print("\nTous les modèles et index RAG sont chargés et prêts !")

# PARAMÈTRE À CHANGER : "bm25", "hyde", "dpr" ou "tf_idf"
APPROACH = "TF-IDF"

approaches_func = {
    "TF-IDF": tf_idf.retrieve,
    #"BM25": bm25.retrieve,
    "HyDE": hyde.retrieve,
    #"DPR": dpr.retrieve
}


Tous les modèles et index RAG sont chargés et prêts !


In [ ]:


def generate_answer(query, retrieved_df,context_k):
    """
    Génère la réponse finale en utilisant Llama et les chunks récupérés.
    """
    # 1. On prépare le texte de contexte avec les 3 meilleurs chunks
    context = [
        "\n---\n".join(retrieved_df["content"].head(context_k).tolist())
    ]
    
    # 2. Formatage du prompt 
    messages = [
                {
                    "role": "system",
                    "content": f"""
            Tu es un assistant RAG francophone.
            
            Ta tâche est de répondre à une question en utilisant uniquement le contexte fourni.
            
            Règles :
            1. Utilise seulement les informations présentes dans le contexte.
            2. N'ajoute aucune information extérieure.
            3. Si la réponse n'est pas explicitement présente ou déductible directement du contexte, réponds :
            "Je ne sais pas d'après le contexte fourni."
            4. Ne mentionne pas que tu es un modèle de langage.
            5. Ne cite pas le contexte mot pour mot sauf si nécessaire.
            6. Réponds de façon courte, précise et naturelle.
            7. Si plusieurs informations du contexte sont utiles, synthétise-les.
            
            Format attendu :
            Une réponse en français, en une ou deux phrases maximum.
            """
                },
                {
                    "role": "user",
                    "content": f"""
            Contexte :
            {context}
            
            Question :
            {query}
            
            Réponse :
            """
                }
    ]
    
    # On réutilise le générateur déjà chargé par hyde.py pour économiser la VRAM du GPU
    prompt = hyde.generator.tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=True
    )
    
    # 3. Inférence
    outputs = hyde.generator(prompt, max_new_tokens=150, return_full_text=False)
    return outputs[0]["generated_text"].strip()

def main():
    print("\n" + "="*60)
    print(f"DÉMARRAGE DE L'ASSISTANT CONVERSATIONNEL RAG (Approche active : {APPROACH.upper()})")
    print("Tapez 'exit', 'quit' ou 'q' pour arrêter.")
    print("="*60)
    
    while True:
        try:
            # Saisie utilisateur
            query = input("\nQuestion : ").strip()
            
            # Condition d'arrêt
            if query.lower() in ["exit", "quit", "q"]:
                print("Arrêt du script")
                break
            if not query:
                continue
            
            rag_results=approaches_func[APPROACH](query)
            
            # --- ÉTAPE 2 : GÉNÉRATION (Réponse) ---
            answer = generate_answer(query, rag_results, 3)
            
            # --- ÉTAPE 3 : AFFICHAGE ---
            print("\n ", answer)
            print("\n Sources utilisées :")
            for i, row in rag_results.head(3).iterrows():
                print(f"   [{i+1}] ({row['url']})")
            print("-" * 60)
            
        except KeyboardInterrupt:
            print("\nArrêt forcé")
            break

if __name__ == "__main__":
    main()


DÉMARRAGE DE L'ASSISTANT CONVERSATIONNEL RAG (Approche active : TF-IDF)
Tapez 'exit', 'quit' ou 'q' pour arrêter.



Question :  c'est quand la conférence


Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



  ][

 Sources utilisées :
   [1] (https://talnarchives.atala.org/TALN/TALN-2025/152.pdf)
   [2] (https://coria-taln-2025.lis-lab.fr/)
   [3] (https://talnarchives.atala.org/TALN/TALN-2025/111.pdf)
------------------------------------------------------------
